# Exercise 2 — Model predictive control of indoor temperature

Same building, outdoor temperature and heater limit as Exercise 1; the room starts at $T_0 = 20$ °C. At every sample $k$ the MPC solves

$$
\begin{aligned}
\min_{u_0,\dots,u_{N-1}}\quad & J = \sum_{i=1}^{N} (r - T_i)^2 + \lambda \sum_{i=0}^{N-1} u_i^2 \\
\text{s.t.}\quad & T_0 = T_k \\
& T_{i+1} = A\,T_i + B\,u_i + E\,\hat T_{o,k+i} \\
& 0 \le u_i \le 2000\ \text{W} \\
& 19 \le T_i \le 24\ ^\circ\text{C}
\end{aligned}
$$

and applies only $u_0$. $\hat T_o$ is the outdoor temperature forecast (assumed perfect).

In [ ]:
# Google Colab only: download the course files and check the packages.
# When you run the notebook on your own computer, this cell does nothing.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("building_model.py"):
        if not os.path.exists("athens_neural_network_for_control"):
            subprocess.run(["git", "clone", "-q", "https://github.com/li-yun/athens_neural_network_for_control.git"], check=True)
        os.chdir("athens_neural_network_for_control")
    r = subprocess.run([sys.executable, "install_packages.py"], capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(out[out.find("2/3"):] if r.returncode == 0 else out)   # show the check results
    if r.returncode != 0:
        raise RuntimeError("Setup failed, see the messages above.")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
from building_model import FirstOrderBuilding, outdoor_temperature, plot_closed_loop, evaluate

model = FirstOrderBuilding()
r, u_max, T0 = 21.0, 2000.0, 20.0
T_min, T_max = 19.0, 24.0
N = 48                                   # horizon: 48 x 30 min = 24 h

To = outdoor_temperature('winter')
To_forecast = np.concatenate([To, np.full(N, To[-1])])   # N extra samples at the end

## Task: write the objective and the constraints

Complete `TODO 1` and `TODO 2`. The rest of the class (parameters, solving, receding horizon) is given.

In [ ]:
class MPC:
    """MPC for the 1R1C building, solved with cvxpy. Call: controller(r_k, T_k, k) -> u_k."""

    def __init__(self, model, N, lam, To_forecast, u_max=2000.0, T_min=19.0, T_max=24.0):
        self.N, self.To_forecast = N, To_forecast
        self.solve_times = []
        A, B, E = model.A, model.B, model.E

        # decision variables
        T = cp.Variable(N + 1)        # T[0] ... T[N]    predicted indoor temperature
        u = cp.Variable(N)            # u[0] ... u[N-1]  heating power
        self.u = u

        # parameters, updated at every time step
        self.T0 = cp.Parameter()      # measured indoor temperature T_k
        self.d = cp.Parameter(N)      # outdoor temperature forecast To[k], ..., To[k+N-1]
        self.r = cp.Parameter(N)      # reference over the horizon

        # TODO 1: objective  J = sum_{i=1..N} (r_i - T_i)^2 + lam * sum_{i=0..N-1} u_i^2
        #         Hint: cp.sum_squares(...), and T[1:] are the predicted temperatures T_1 ... T_N
        J = ...

        # TODO 2: constraints (a list)
        #   T[0] == T0
        #   T[i+1] == A T[i] + B u[i] + E d[i]   ->  in vector form with T[1:] and T[:-1]
        #   0 <= u <= u_max
        #   T_min <= T[1:] <= T_max
        constraints = [...]

        self.prob = cp.Problem(cp.Minimize(J), constraints)

    def __call__(self, r_k, T_k, k):
        self.T0.value = T_k
        self.d.value = self.To_forecast[k:k + self.N]
        self.r.value = np.full(self.N, r_k)
        t0 = time.perf_counter()
        self.prob.solve(solver=cp.CLARABEL)
        self.solve_times.append(time.perf_counter() - t0)
        if self.prob.status != cp.OPTIMAL:
            raise RuntimeError(f"MPC problem at k={k}: status {self.prob.status}")
        return float(np.clip(self.u.value[0], 0.0, None))   # apply only the first input

In [ ]:
# Check: solve once at k = 0
mpc = MPC(model, N, lam=1e-6, To_forecast=To_forecast)
print(f'u_0 = {mpc(r, T0, 0):.0f} W, status = {mpc.prob.status}')

## Closed loop: control performance and solve time

In [ ]:
mpc = MPC(model, N, lam=1e-6, To_forecast=To_forecast)
res_mpc = model.simulate_closed_loop(T0, r, To, controller=mpc, pass_k=True)
plot_closed_loop(res_mpc, labels=['MPC, λ = 1e-6'], u_max=u_max, T_band=(T_min, T_max))
plt.show()

t_ms = np.array(mpc.solve_times[1:]) * 1e3      # first solve includes cvxpy compilation
print(evaluate(res_mpc, T_min=T_min, T_max=T_max))
print(f'solve time: mean {t_ms.mean():.2f} ms, max {t_ms.max():.2f} ms  (sampling time = 30 min)')

## Comparison with the P controller of Exercise 1

In [ ]:
def p_controller(r, T, Kp, u_max):
    return np.clip(Kp * (r - T), 0.0, u_max)

def timed(controller, times):
    def wrapped(r_k, T_k, k):
        t0 = time.perf_counter(); u = controller(r_k, T_k, k); times.append(time.perf_counter() - t0)
        return u
    return wrapped

cases = {
    'P,   Kp = 1000': lambda r_k, T_k, k: p_controller(r_k, T_k, 1000, u_max),
    'P,   Kp = 5000': lambda r_k, T_k, k: p_controller(r_k, T_k, 5000, u_max),
    'MPC, λ = 1e-6': MPC(model, N, lam=1e-6, To_forecast=To_forecast),
    'MPC, λ = 1e-4': MPC(model, N, lam=1e-4, To_forecast=To_forecast),
}
rows, results = {}, []
for name, ctrl in cases.items():
    times = []
    res = model.simulate_closed_loop(T0, r, To, controller=timed(ctrl, times), pass_k=True)
    results.append(res)
    rows[name] = {**evaluate(res, T_min=T_min, T_max=T_max), 'time per step [ms]': 1e3 * np.mean(times[1:])}

plot_closed_loop(results, labels=list(cases), u_max=u_max, T_band=(T_min, T_max))
plt.show()
pd.DataFrame(rows).T.round(4)